<a href="https://colab.research.google.com/github/shfeekah/Uncertainty-Markov-Chains-and-Propositional-Logic/blob/main/lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Task 1: Toy Weather Graph


In [ ]:
import numpy as np
import pandas as pd

states = ['Sunny', 'Cloudy', 'Rainy']
init = {'Sunny': 0.5, 'Cloudy': 0.3, 'Rainy': 0.2}
T = {
    'Sunny':  {'Sunny': 0.6, 'Cloudy': 0.3, 'Rainy': 0.1},
    'Cloudy': {'Sunny': 0.3, 'Cloudy': 0.4, 'Rainy': 0.3},
    'Rainy':  {'Sunny': 0.2, 'Cloudy': 0.4, 'Rainy': 0.4},
}

# 1. Joint distribution P(today, tomorrow) = P(today) * P(tomorrow | today)
joint_df = pd.DataFrame(index=states, columns=states, dtype=float)
for s1 in states:
    for s2 in states:
        joint_df.loc[s1, s2] = init[s1] * T[s1][s2]
print(joint_df)
print("Sum of all joint probabilities:", joint_df.values.sum())

        Sunny  Cloudy  Rainy
Sunny    0.30    0.15   0.05
Cloudy   0.09    0.12   0.09
Rainy    0.04    0.08   0.08
Sum of all joint probabilities: 1.0000000000000002


In [ ]:
# 2. Marginals
p_tomorrow = joint_df.sum(axis=0)
p_today = joint_df.sum(axis=1)
print(p_tomorrow)


Sunny     0.43
Cloudy    0.35
Rainy     0.22
dtype: float64


In [ ]:
# 3. Conditionals computed from the joint table
p_tomorrow_given_today_rainy = joint_df.loc['Rainy'] / p_today['Rainy']
p_today_given_tomorrow_rainy = joint_df['Rainy'] / p_tomorrow['Rainy']
print(p_tomorrow_given_today_rainy)
print(p_today_given_tomorrow_rainy)

Sunny     0.2
Cloudy    0.4
Rainy     0.4
Name: Rainy, dtype: float64
Sunny     0.227273
Cloudy    0.409091
Rainy     0.363636
Name: Rainy, dtype: float64


In [ ]:
# 4. Bayes' rule
bayes = (T['Rainy']['Rainy'] * init['Rainy']) / p_tomorrow['Rainy']
print(f"P(today=Rainy | tomorrow=Rainy) = {bayes:.4f}")

P(today=Rainy | tomorrow=Rainy) = 0.3636


In [ ]:
# 5. Independence check
p_ss = joint_df.loc['Sunny', 'Sunny']
p_ind = p_today['Sunny'] * p_tomorrow['Sunny']
print(p_ss, p_ind, np.isclose(p_ss, p_ind))

0.3 0.215 False


Task 2: Real Data (Breast Cancer Diagnostic Dataset)

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

df = load_breast_cancer(as_frame=True).frame
df['malignant'] = (df['target'] == 0)   # 0 means malignant
df['large'] = df['mean radius'] > df['mean radius'].median()
df['rough'] = df['mean texture'] > df['mean texture'].median()

# 1. Explore
print("Shape:", df.shape)
print("Malignant:", df['malignant'].sum())
print("Benign:", (~df['malignant']).sum())

Shape: (569, 34)
Malignant: 212
Benign: 357


In [ ]:

# 2. Joint distribution
joint = pd.crosstab(df['malignant'], df['large'], normalize='all')
print(joint)
print("Sum:", joint.values.sum())

large         False     True 
malignant                    
False      0.471002  0.156415
True       0.029877  0.342707
Sum: 1.0


In [ ]:
# 3. Probabilities from the table
p_mal = df['malignant'].mean()
p_large = df['large'].mean()
p_large_given_mal = joint.loc[True, True] / joint.loc[True].sum()
p_large_given_ben = joint.loc[False, True] / joint.loc[False].sum()


In [ ]:
# 4. Bayes' rule vs direct counting
p_mal_given_large_bayes = (p_large_given_mal * p_mal) / p_large
p_mal_given_large_direct = df[df['large']]['malignant'].mean()
p_mal_given_not_large = df[~df['large']]['malignant'].mean()

In [ ]:
# 5. Independence
p_joint = joint.loc[True, True]
print(p_joint, p_mal * p_large)

0.3427065026362039 0.18596433789122221


Topic 2: Markov Chains and Language Model

Part A: Weather Chain

In [ ]:
import numpy as np

states = ['Sunny', 'Cloudy', 'Rainy']
P = np.array([
    [0.6, 0.3, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.4, 0.4]
])

# 1. Simulate 10 days starting from Sunny
np.random.seed(42)
curr = 0
seq = [states[curr]]
for _ in range(9):
    curr = np.random.choice([0, 1, 2], p=P[curr])
    seq.append(states[curr])
print(seq)


['Sunny', 'Sunny', 'Rainy', 'Rainy', 'Cloudy', 'Sunny', 'Sunny', 'Sunny', 'Cloudy', 'Cloudy']


In [ ]:
# 2. Distribution after 2 and 30 days
v = np.array([1.0, 0.0, 0.0])
print(v @ np.linalg.matrix_power(P, 2))
print(v @ np.linalg.matrix_power(P, 30))



[0.47 0.34 0.19]
[0.39344262 0.36065574 0.24590164]


In [ ]:
# 3. 100 steps from two different starting points
print(v @ np.linalg.matrix_power(P, 100))
print(np.array([0.0, 0.0, 1.0]) @ np.linalg.matrix_power(P, 100))

[0.39344262 0.36065574 0.24590164]
[0.39344262 0.36065574 0.24590164]


Part B: Word Graph / Bigram Model

In [ ]:
from collections import defaultdict
import numpy as np

corpus = [
    "the student writes the code.",
    "the student runs the code.",
    "the code prints the result.",
    "the teacher checks the code.",
]

# 1. Tokenize (add <s> and </s>) and count words and bigrams
def tokenize(sentence):
    return ["<s>"] + sentence.replace('.', ' .').split() + ["</s>"]

word_counts = defaultdict(int)
bigram_counts = defaultdict(lambda: defaultdict(int))
for sent in [tokenize(s) for s in corpus]:
    for i, w in enumerate(sent):
        word_counts[w] += 1
        if i < len(sent) - 1:
            bigram_counts[w][sent[i + 1]] += 1



In [ ]:
# 2. P(next | current) = count(pair) / count(current)
bigram_probs = defaultdict(dict)
for w, nxt in bigram_counts.items():
    for w2, c in nxt.items():
        bigram_probs[w][w2] = c / word_counts[w]
for w2, p in bigram_probs['the'].items():
    print(f"P({w2} | the) = {p:.3f}")


P(student | the) = 0.250
P(code | the) = 0.500
P(result | the) = 0.125
P(teacher | the) = 0.125


In [ ]:
 #3. Sentence probability
def sentence_prob(sentence):
    t = tokenize(sentence)
    prob = 1.0
    for a, b in zip(t, t[1:]):
        prob *= bigram_probs[a].get(b, 0.0)
    return prob

print(sentence_prob("the student runs the code."))
print(sentence_prob("the teacher runs the code."))



0.046875
0.0


In [ ]:
# 4. Generate 3 sentences
np.random.seed(42)
for i in range(1, 4):
    curr, tokens = "<s>", []
    while curr != "</s>":
        words = list(bigram_probs[curr].keys())
        probs = list(bigram_probs[curr].values())
        curr = np.random.choice(words, p=probs)
        if curr != "</s>":
            tokens.append(curr)
    print(f"Sentence {i}:", ' '.join(tokens))



Sentence 1: the teacher checks the student writes the result .
Sentence 2: the teacher checks the student writes the code .
Sentence 3: the student writes the code prints the code .
